In [1]:
!pip install imbalanced-learn --quiet

In [2]:
import imblearn
print(imblearn.__version__)

0.14.0


In [3]:
# ============================================
# PHASE 3 — Class-Aware Performance & Imbalance Handling (RQ3)
# CICIDS2017 — Baseline vs SMOTE vs Undersampling vs Class Weighting
# ============================================

import pandas as pd
import numpy as np
import time
import os
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, matthews_corrcoef, confusion_matrix)
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# ---- 1. LOAD DATA (same as Phase 1 / Unseen Testing) ----
train_df = pd.read_csv(r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\CIC-IDS2017\processed\CICIDS2017_train_cleaned.csv")
test_df = pd.read_csv(r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\CIC-IDS2017\processed\CICIDS2017_test_cleaned.csv")

X_train = train_df.drop(columns=['Label', 'binary_label'])
y_train = train_df['binary_label']
X_test = test_df.drop(columns=['Label', 'binary_label'])
y_test = test_df['binary_label']

POS_LABEL = 'ATTACK'
LABELS = ['BENIGN', 'ATTACK']

print("Original training class balance:")
print(y_train.value_counts())

# ---- 2. PREPARE THE FOUR CONDITIONS ----
print("\nApplying SMOTE (this may take a few minutes on ~2M rows)...")
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print("SMOTE done. New class balance:", pd.Series(y_train_smote).value_counts().to_dict())

print("\nApplying random undersampling...")
undersampler = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = undersampler.fit_resample(X_train, y_train)
print("Undersampling done. New class balance:", pd.Series(y_train_under).value_counts().to_dict())

# ---- 3. MODELS (baseline + class-weighted versions) ----
def get_models(class_weight=None):
    return {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight=class_weight),
        'KNN': KNeighborsClassifier(n_jobs=-1),  # KNN has no class_weight parameter
        'Decision Tree': DecisionTreeClassifier(random_state=42, class_weight=class_weight),
        'Random Forest': RandomForestClassifier(random_state=42, n_jobs=-1, class_weight=class_weight),
        'SVM': LinearSVC(random_state=42, max_iter=2000, class_weight=class_weight)
    }

# ---- 4. EVALUATION FUNCTION ----
def evaluate_condition(X_tr, y_tr, X_te, y_te, models, condition_name):
    results = []
    print(f"\n{'='*50}\nCONDITION: {condition_name}\n{'='*50}")
    for name, model in models.items():
        start = time.time()
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)
        elapsed = time.time() - start

        acc = accuracy_score(y_te, y_pred)
        prec = precision_score(y_te, y_pred, pos_label=POS_LABEL)
        rec = recall_score(y_te, y_pred, pos_label=POS_LABEL)
        f1 = f1_score(y_te, y_pred, pos_label=POS_LABEL)
        mcc = matthews_corrcoef(y_te, y_pred)
        tn, fp, fn, tp = confusion_matrix(y_te, y_pred, labels=LABELS).ravel()
        fpr = fp / (fp + tn)

        print(f"{name}: acc={acc:.4f}, prec={prec:.4f}, rec={rec:.4f}, f1={f1:.4f}, mcc={mcc:.4f} ({elapsed:.1f}s)")

        results.append({
            'condition': condition_name, 'model': name, 'accuracy': acc,
            'precision': prec, 'recall': rec, 'f1': f1, 'fpr': fpr, 'mcc': mcc
        })
    return results

# ---- 5. RUN ALL FOUR CONDITIONS ----
all_results = []

# Baseline (no imbalance handling)
all_results += evaluate_condition(X_train, y_train, X_test, y_test, get_models(), "Baseline (No Handling)")

# SMOTE
all_results += evaluate_condition(X_train_smote, y_train_smote, X_test, y_test, get_models(), "SMOTE")

# Undersampling
all_results += evaluate_condition(X_train_under, y_train_under, X_test, y_test, get_models(), "Random Undersampling")

# Class Weighting (trained on original imbalanced data, but with class_weight='balanced')
all_results += evaluate_condition(X_train, y_train, X_test, y_test, get_models(class_weight='balanced'), "Class Weighting")

# ---- 6. SAVE RESULTS ----
results_df = pd.DataFrame(all_results)
results_folder = r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\Phase3_results"
os.makedirs(results_folder, exist_ok=True)
results_df.to_csv(os.path.join(results_folder, "Phase3_CICIDS2017_imbalance_comparison.csv"), index=False)

print("\n\nAll Phase 3 results saved successfully!")
print(os.path.join(results_folder, "Phase3_CICIDS2017_imbalance_comparison.csv"))

Original training class balance:
binary_label
BENIGN    1676045
ATTACK     340593
Name: count, dtype: int64

Applying SMOTE (this may take a few minutes on ~2M rows)...
SMOTE done. New class balance: {'BENIGN': 1676045, 'ATTACK': 1676045}

Applying random undersampling...
Undersampling done. New class balance: {'ATTACK': 340593, 'BENIGN': 340593}

CONDITION: Baseline (No Handling)
Logistic Regression: acc=0.9571, prec=0.8809, rec=0.8627, f1=0.8717, mcc=0.8460 (93.2s)
KNN: acc=0.9954, prec=0.9841, rec=0.9889, f1=0.9865, mcc=0.9837 (4713.3s)
Decision Tree: acc=0.9986, prec=0.9957, rec=0.9959, f1=0.9958, mcc=0.9950 (195.9s)
Random Forest: acc=0.9986, prec=0.9963, rec=0.9951, f1=0.9957, mcc=0.9949 (149.7s)
SVM: acc=0.9529, prec=0.8710, rec=0.8463, f1=0.8585, mcc=0.8303 (734.3s)

CONDITION: SMOTE
Logistic Regression: acc=0.9404, prec=0.7477, rec=0.9766, f1=0.8469, mcc=0.8219 (228.0s)
KNN: acc=0.9950, prec=0.9752, rec=0.9956, f1=0.9853, mcc=0.9824 (3557.8s)
Decision Tree: acc=0.9987, prec=0.